In [ ]:
'''
BT 4 - Validation and reflection.ipynb
Author: Muhammmad Talha & Jingchuan Shi
Supervisor: Dr. Ahmed Qureshi
Created 2019/9/6, last modified 2026/02/02 at University of Alberta.
All Rights Reserved.
'''

# Load relevant modules.
from __future__ import annotations
import os, ast, json, math, time, random, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# nlp = spacy.load("en_core_web_lg")          # ← replaces "en_vectors"

In [2]:
# The list of verbs pre-labelled with corresponding Bloom's Taxonomy domains.
knowledge_words = ['list', 'name', 'define', 'repeat', 'state', 'label', 'recall', 'identify', 'reproduce', 'describe', 'recognize', 'select', 'record', 'match', 'relate', 'memorize', 'outline', 'quote', 'enumerate', 'write', 'tell', 'recite', 'cite', 'duplicate', 'read', 'order', 'tabulate', 'draw', 'review', 'indicate', 'underline', 'arrange', 'know', 'point', 'count', 'collect', 'meet', 'study', 'trace', 'find', 'index', 'locate', 'show', 'visualize', 'examine', 'copy', 'sequence', 'acquire', 'retell', 'view', 'observe', 'tally', 'imitate', 'follow']
comprehension_words = ['explain', 'describe', 'discuss', 'paraphrase', 'restate', 'summarize', 'translate', 'convert', 'review', 'express', 'estimate', 'identify', 'generalize', 'interpret', 'locate', 'give', 'distinguish', 'extend', 'predict', 'recognize', 'defend', 'classify', 'infer', 'report', 'illustrate', 'rewrite', 'select', 'contrast', 'differentiate', 'compare', 'indicate', 'exemplify', 'observe', 'elaborate', 'associate', 'visualize', 'articulate', 'clarify', 'subtract', 'approximate', 'interpolate', 'tell', 'detail', 'outline', 'cite', 'picture', 'interact', 'conclude', 'characterize', 'add', 'factor', 'compute', 'match', 'schedule', 'order', 'sketch', 'draw', 'define', 'operate', 'arrange', 'group', 'extrapolate', 'diagram', 'interrelate', 'represent', 'trace', 'shop', 'suggest', 'understand']
application_words = ['demonstrate', 'use', 'apply', 'solve', 'illustrate', 'dramatize', 'practise', 'employ', 'operate', 'sketch', 'prepare', 'show', 'compute', 'relate', 'construct', 'interpret', 'discover', 'change', 'produce', 'manipulate', 'schedule', 'modify', 'predict', 'complete', 'choose', 'classify', 'translate', 'determine', 'examine', 'calculate', 'investigate', 'draw', 'write', 'protect', 'derive', 'chart', 'alphabetize', 'simulate', 'process', 'provide', 'capture', 'project', 'transcribe', 'organize', 'shop', 'establish', 'attain', 'graph', 'assign', 'allocate', 'convert', 'experiment', 'exercise', 'diminish', 'make', 'develop', 'ascertain', 'tabulate', 'depreciate', 'subscribe', 'implement', 'handle', 'transfer', 'factor', 'avoid', 'expose', 'express', 'perform', 'sequence', 'acquire', 'administer', 'personalize', 'adapt', 'plot', 'customize', 'interview', 'paint', 'explore', 'utilize', 'report', 'figure', 'price', 'coordinate', 'simplify', 'consult', 'maintain', 'deliver', 'extend', 'imitate', 'guide', 'conduct', 'multiply', 'build', 'code', 'contribute', 'obtain', 'model', 'compare', 'divide', 'exhibit', 'tally', 'inform', 'diagram', 'expand', 'amend', 'engineer', 'control', 'assess', 'concatenate', 'execute', 'convey', 'articulate', 'restructure', 'criticize', 'appraise', 'participate', 'generalize', 'instruct', 'follow', 'act', 'screen', 'debate', 'question', 'select', 'include', 'dissect', 'retrieve', 'inspect', 'prove', 'inventory', 'respond', 'comply', 'collect']
analysis_words = ['compare', 'contrast', 'distinguish', 'analyze', 'differentiate', 'separate', 'examine', 'diagram', 'infer', 'categorize', 'experiment', 'discriminate', 'select', 'appraise', 'relate', 'test', 'question', 'classify', 'identify', 'outline', 'illustrate', 'subdivide', 'investigate', 'debate', 'criticize', 'calculate', 'inventory', 'prioritize', 'correlate', 'explain', 'inspect', 'detect', 'dissect', 'manage', 'audit', 'characterize', 'order', 'deduce', 'limit', 'connect', 'diagnose', 'document', 'proofread', 'discover', 'ensure', 'optimize', 'maximize', 'confirm', 'divide', 'transform', 'figure', 'prepare', 'file', 'determine', 'train', 'solve', 'survey', 'group', 'minimize', 'interrupt', 'explore', 'blueprint', 'arrange', 'query', 'edit', 'prove', 'isolate', 'reconcile', 'troubleshoot', 'sketch', 'create', 'summarize', 'dramatize', 'employ', 'inquire', 'link', 'abstract', 'establish', 'organize', 'compute', 'devise', 'moderate', 'delegate', 'research', 'model', 'practise', 'operate', 'demonstrate', 'schedule', 'check', 'use', 'chunk', 'choose', 'scrutinize', 'chart', 'apply', 'allow', 'extrapolate', 'recognize', 'show', 'modify', 'administer', 'review', 'change', 'monitor', 'direct', 'corroborate', 'produce', 'negotiate', 'probe', 'accept', 'design', 'interpret', 'extract', 'manipulate', 'focus', 'write', 'predict', 'resolve']
synthesis_words = ['design', 'create', 'formulate', 'plan', 'compose', 'construct', 'develop', 'combine', 'assemble', 'propose', 'devise', 'arrange', 'organize', 'collect', 'rearrange', 'prepare', 'reconstruct', 'invent', 'generate', 'modify', 'write', 'categorize', 'rewrite', 'relate', 'compile', 'revise', 'reorganize', 'summarize', 'manage', 'generalize', 'integrate', 'explain', 'produce', 'originate', 'tell', 'incorporate', 'facilitate', 'hypothesize', 'substitute', 'specify', 'improve', 'format', 'correspond', 'model', 'depict', 'synthesize', 'refer', 'comply', 'enhance', 'import', 'overhaul', 'animate', 'predict', 'adapt', 'cultivate', 'code', 'join', 'handle', 'anticipate', 'portray', 'express', 'budget', 'cope', 'debug', 'perform', 'communicate', 'outline', 'prescribe', 'initiate', 'network', 'program', 'lecture', 'dictate', 'advise', 'document', 'gather', 'derive', 'abstract', 'expand', 'establish', 'collaborate', 'conduct', 'contribute', 'coordinate', 'compare', 'speculate', 'simulate', 'progress', 'forecast', 'instruct', 'structure', 'intervene', 'frame', 'measure', 'estimate', 'recommend', 'negotiate', 'consolidate', 'choose', 'contrast', 'imagine', 'individualize', 'recognize', 'solve', 'roleplay', 'review', 'arbitrate', 'teach', 'supervise', 'assess', 'counsel', 'exchange', 'brief', 'reinforce', 'unify', 'pretend', 'update', 'validate']
evaluation_words = ['judge', 'appraise', 'evaluate', 'support', 'assess', 'select', 'justify', 'compare', 'rate', 'conclude', 'value', 'defend', 'estimate', 'choose', 'critique', 'argue', 'measure', 'recommend', 'discriminate', 'decide', 'interpret', 'criticize', 'contrast', 'rank', 'predict', 'explain', 'summarize', 'score', 'grade', 'revise', 'relate', 'verify', 'test', 'validate', 'attach', 'determine', 'describe', 'convince', 'prescribe', 'consider', 'release', 'counsel', 'hire', 'prioritize', 'deduce', 'enforce', 'advise', 'motivate', 'core', 'uphold', 'resolve', 'reconcile', 'discuss', 'authenticate', 'review', 'monitor', 'weigh', 'debate', 'diagnose', 'infer', 'mediate', 'prove', 'use', 'preserve', 'access', 'consolidate']
wordlists = [knowledge_words, comprehension_words, application_words, analysis_words, synthesis_words, evaluation_words]
namelist = ['knowledge', 'comprehension', 'application', 'analysis', 'synthesis', 'evaluation']

BASE_DIR    = Path.cwd()
RESULTS_DIR = BASE_DIR / "results"
FIG_DIR     = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]

SRC = Path("results/BT4_structural_best/BT4_PROD_TUNING_SGDClassifier_P60_2025-10-12_01-41-14.csv") #or the file with your classified verbs, in our BT3 it was saved with this name
assert SRC.exists(), f"CSV not found at {SRC.resolve()}"

df = pd.read_csv(SRC, encoding="utf-8")


# --- Normalise verb column name so later code can assume "verb" exists ---
if "verb" not in df.columns:
    for cand in ["Verb_clean", "Verb", "VERB", "word", "Word"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "verb"})
            break
    else:
        raise ValueError(f"No verb-like column found. Columns are: {list(df.columns)}")

# Now safely clean the verb column
df["verb"] = df["verb"].astype(str).str.strip().str.lower()

# Make sure one-hots are ints, if present
for lv in LEVELS:
    if lv in df.columns:
        df[lv] = pd.to_numeric(df[lv], errors="coerce").fillna(0).astype(int)

# Make sure accept_* are ints, if present
for lv in LEVELS:
    col = f"accept_{lv}"
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype(int)

has_accept = all(f"accept_{lv}" in df.columns for lv in LEVELS)
has_count  = "AcceptedCount" in df.columns



In [ ]:
# ===  stats (using CSV columns as-is) === (for validation with BT3)
n = len(df)

# Acceptance (hard) straight from your field(s)
if "Accepted" in df.columns:
    accepted_words = int(pd.to_numeric(df["Accepted"], errors="coerce").fillna(0).astype(int).sum())
else:
    accepted_words = int((pd.to_numeric(df.get("AcceptedCount", 0), errors="coerce").fillna(0) > 0).sum())

rejected_words = n - accepted_words
acc_rate = accepted_words / n if n else 0.0

# Entry rate: prefer AcceptedCount; else sum(LEVEL one-hots)
if has_count:
    entry_rate = float(pd.to_numeric(df["AcceptedCount"], errors="coerce").fillna(0).astype(int).mean())
else:
    entry_rate = float(df[LEVELS].sum(axis=1).mean())

# Fully accepted (all 6)
if has_accept:
    fully_accepted = int((df[[f"accept_{lv}" for lv in LEVELS]].sum(axis=1) == 6).sum())
else:
    fully_accepted = int((df[LEVELS].sum(axis=1) == 6).sum())

# Single vs multi among hard-accepted words
label_counts = (df[LEVELS].sum(axis=1) if all(lv in df.columns for lv in LEVELS)
                else pd.to_numeric(df.get("AcceptedCount", 0), errors="coerce").fillna(0).astype(int))
accepted_mask = (pd.to_numeric(df["Accepted"], errors="coerce").fillna(0).astype(int) == 1) if "Accepted" in df.columns else (label_counts > 0)
single_label_items = int(((label_counts == 1) & accepted_mask).sum())
multi_label_items  = int(((label_counts >= 2) & accepted_mask).sum())

# Suggested-only (only if you already have soft fields)
if "SoftAcceptedCount" in df.columns:
    suggested_only = int(((pd.to_numeric(df.get("Accepted", 0), errors="coerce").fillna(0).astype(int) == 0) &
                          (pd.to_numeric(df["SoftAcceptedCount"], errors="coerce").fillna(0).astype(int) > 0)).sum())
elif "SoftAccepted" in df.columns:
    suggested_only = int(((pd.to_numeric(df.get("Accepted", 0), errors="coerce").fillna(0).astype(int) == 0) &
                          (pd.to_numeric(df["SoftAccepted"], errors="coerce").fillna(0).astype(int) == 1)).sum())
else:
    suggested_only = 0

# Distribution by #entries per word
if has_count:
    counts_by_entries = pd.to_numeric(df["AcceptedCount"], errors="coerce").fillna(0).astype(int).value_counts().reindex(range(0,7), fill_value=0).astype(int).to_dict()
elif all(lv in df.columns for lv in LEVELS):
    counts_by_entries = label_counts.value_counts().reindex(range(0,7), fill_value=0).astype(int).to_dict()
else:
    counts_by_entries = {}

# Per-domain **entries** (BT3-style): sum of accept_* if available, else sum of one-hots
if has_accept:
    per_domain_entries = {lv: int(df[f"accept_{lv}"].sum()) for lv in LEVELS}
elif all(lv in df.columns for lv in LEVELS):
    per_domain_entries = {lv: int(df[lv].sum()) for lv in LEVELS}
else:
    per_domain_entries = {}

# Verb-level **counts** per PrimaryDomain exactly as in file (no mapping/normalization)
prim = df.get("PrimaryDomain", pd.Series([], dtype=object)).astype(str)
verb_counts_primarydomain = prim.value_counts(dropna=False)

# Threshold min/max if present
thr_cols = [f"thr_{lv}" for lv in LEVELS if f"thr_{lv}" in df.columns]
thr_min = float(min(pd.to_numeric(df[c], errors="coerce").min() for c in thr_cols)) if thr_cols else None
thr_max = float(max(pd.to_numeric(df[c], errors="coerce").max() for c in thr_cols)) if thr_cols else None

print(f"Acceptance rate: {acc_rate:.6f}   (accepted {accepted_words} / {n}, rejected {rejected_words})")
print(f"Entry rate:      {entry_rate:.6f}")
print(f"Fully accepted (all 6): {fully_accepted}")
print(f"Single-label items:     {single_label_items}")
print(f"Multi-label items:      {multi_label_items}")
print(f"Suggested-only (soft>0, hard=0): {suggested_only}")
if thr_cols:
    print(f"thr min/max: {thr_min:.6f} / {thr_max:.6f}")

print("\n# of words with k entries:")
for k in range(0,7):
    print(f"  {k}: {counts_by_entries.get(k,0)}")

print("\nEntries acquired by domain (BT3-style label entries):")
for lv in LEVELS:
    print(f"  {lv}: {per_domain_entries.get(lv, 0)}")

print("\nVerb counts by PrimaryDomain (exact values from file, including blanks):")
print(verb_counts_primarydomain.head(20))


Acceptance rate: 0.825658   (accepted 1506 / 1824, rejected 318)
Entry rate:      1.019189
Fully accepted (all 6): 0
Single-label items:     477
Multi-label items:      461
Suggested-only (soft>0, hard=0): 240
thr min/max: 0.637883 / 0.910864

# of words with k entries:
  0: 318
  1: 1246
  2: 185
  3: 61
  4: 10
  5: 4
  6: 0

Entries acquired by domain (BT3-style label entries):
  Kn: 49
  Cm: 373
  Ap: 574
  An: 259
  Sn: 485
  Ev: 119

Verb counts by PrimaryDomain (exact values from file, including blanks):
PrimaryDomain
Ap     467
Sn     407
nan    318
Cm     301
An     190
Ev     100
Kn      41
Name: count, dtype: int64


In [5]:
from pathlib import Path

def sample_expert_from_csv(
    dfx: pd.DataFrame,
    n_per_domain: int = 5,
    out_name: str = "ExpertSample_single.csv",
    seed: int = 42
) -> Path:
    assert "PrimaryDomain" in dfx.columns, "PrimaryDomain column required in the CSV."
    assert "Accepted" in dfx.columns, "Accepted column required in the CSV for hard-only sampling."

    dfx = dfx.copy()

    # Hard-accepted only
    dfx["Accepted"] = pd.to_numeric(dfx["Accepted"], errors="coerce").fillna(0).astype(int)
    dfx_hard = dfx[dfx["Accepted"] == 1].copy()

    # Drop blanks/NaN primary domain *as-is* (no mapping)
    pd_nonblank = dfx_hard["PrimaryDomain"].astype(str).str.strip()
    mask_nonblank = pd_nonblank.ne("") & pd_nonblank.ne("nan")
    dfx_hard = dfx_hard[mask_nonblank]

    # Sample exactly n per PrimaryDomain string in the file
    parts = []
    rng = np.random.RandomState(seed)
    for dom, g in dfx_hard.groupby("PrimaryDomain", dropna=False):
        k = min(n_per_domain, len(g))
        if k > 0:
            parts.append(g.sample(n=k, random_state=rng))
    if not parts:
        raise ValueError("No rows sampled. Check Accepted==1 and PrimaryDomain values.")

    out_df = pd.concat(parts, ignore_index=True)

    # provenance columns (prepend), keep ALL original columns untouched
    out_df.insert(0, "domain_sampled", out_df["PrimaryDomain"])
    out_df.insert(0, "sample_origin", "per_PrimaryDomain_hard")

    out_path = Path("results") / out_name
    out_df.to_csv(out_path, index=False, encoding="utf-8")
    print(f"Saved: {out_path} (rows={len(out_df)})")
    return out_path

expert_csv = sample_expert_from_csv(df, n_per_domain=5, out_name="ExpertSample_single.csv", seed=51)
expert_csv


Saved: results\ExpertSample_single.csv (rows=30)


WindowsPath('results/ExpertSample_single.csv')

In [6]:
# === Figures: entries per level & histogram of labels per verb ===
def figure_label_counts(df: pd.DataFrame) -> Path:
    sums = [int(df[lv].sum()) for lv in LEVELS]
    plt.figure(figsize=(7,4))
    plt.bar(LEVELS, sums)
    plt.title("Entries per Cognitive Level")
    plt.xlabel("Level"); plt.ylabel("# Entries")
    out = FIG_DIR / "fig_entries_per_level.png"
    plt.tight_layout(); plt.savefig(out, dpi=200); plt.close()
    print("Saved:", out); return out

def figure_nlabels_hist(df: pd.DataFrame) -> Path:
    nlab = df[LEVELS].sum(axis=1)
    vals = nlab.value_counts().sort_index()
    plt.figure(figsize=(7,4))
    plt.bar(vals.index.astype(int), vals.values)
    plt.title("# of Cognitive Levels per Verb")
    plt.xlabel("# levels per verb"); plt.ylabel("# verbs")
    out = FIG_DIR / "fig_nlabels_hist.png"
    plt.tight_layout(); plt.savefig(out, dpi=200); plt.close()
    print("Saved:", out); return out

figure_label_counts(df)
figure_nlabels_hist(df)


Saved: D:\Downloads\Bloom-s-Taxonomy-extension-master\Bloom-s-Taxonomy-extension-master\results\figures\fig_entries_per_level.png
Saved: D:\Downloads\Bloom-s-Taxonomy-extension-master\Bloom-s-Taxonomy-extension-master\results\figures\fig_nlabels_hist.png


WindowsPath('D:/Downloads/Bloom-s-Taxonomy-extension-master/Bloom-s-Taxonomy-extension-master/results/figures/fig_nlabels_hist.png')

In [7]:
# === Mutual Influence Heatmap ===
def figure_mutual_influence_heatmap(df: pd.DataFrame) -> Path:
    total = len(df)
    B = df[LEVELS].to_numpy(dtype=int)
    subtotals = B.sum(axis=0)
    M = np.full((6,6), np.nan, dtype=float)
    for i in range(6):
        for j in range(6):
            if i == j: continue
            count_ij = int((B[:,i] & B[:,j]).sum())
            denom = max(int(subtotals[i]) * int(subtotals[j]), 1)
            M[i,j] = (count_ij * total) / denom

    plt.figure(figsize=(6,5))
    im = plt.imshow(M, interpolation="nearest")
    plt.colorbar(im, fraction=0.046, pad=0.04)
    plt.xticks(range(6), LEVELS); plt.yticks(range(6), LEVELS)
    plt.title("Mutual Influence Matrix")
    out = FIG_DIR / "fig_mutual_influence_heatmap.png"
    plt.tight_layout(); plt.savefig(out, dpi=200); plt.close()
    print("Saved:", out); return out

figure_mutual_influence_heatmap(df)


Saved: D:\Downloads\Bloom-s-Taxonomy-extension-master\Bloom-s-Taxonomy-extension-master\results\figures\fig_mutual_influence_heatmap.png


WindowsPath('D:/Downloads/Bloom-s-Taxonomy-extension-master/Bloom-s-Taxonomy-extension-master/results/figures/fig_mutual_influence_heatmap.png')

In [8]:
# === Embedding backend (Torch ST -> TF-IDF+SVD fallback) ===
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

_ST_MODEL = None
_VECT_BACKEND = None
_SVD_PIPE = None

def _try_load_st():
    global _ST_MODEL, _VECT_BACKEND
    if _ST_MODEL is not None: return _ST_MODEL
    try:
        from sentence_transformers import SentenceTransformer
        _ST_MODEL = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        _VECT_BACKEND = "st"
        return _ST_MODEL
    except Exception as e:
        warnings.warn(f"Sentence-Transformers unavailable, using TF-IDF+SVD fallback: {e}")
        _ST_MODEL = None; _VECT_BACKEND = None
        return None

def _fit_svd(words: List[str], n_components: int = 128):
    global _SVD_PIPE, _VECT_BACKEND
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.decomposition import TruncatedSVD
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import Normalizer
    vect = TfidfVectorizer(analyzer="word", lowercase=True, token_pattern=r"(?u)\b\w+\b")
    svd  = TruncatedSVD(n_components=min(n_components, max(2, len(words)-1)))
    norm = Normalizer(copy=False)
    _SVD_PIPE = make_pipeline(vect, svd, norm).fit(words)
    _VECT_BACKEND = "svd"

def embed_words(words: List[str]) -> np.ndarray:
    mdl = _try_load_st()
    W = [w.strip().lower() for w in words]
    if mdl is not None:
        V = mdl.encode(W, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32, copy=False)
        n = np.linalg.norm(V, axis=1, keepdims=True) + 1e-12
        return V / n
    if _SVD_PIPE is None: _fit_svd(W)
    X = _SVD_PIPE.transform(W).astype(np.float32, copy=False)
    n = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12
    return X / n


In [9]:
# === FIXED 2D visualization (robust text column + consistent legend colors) ===
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

LEVELS  = ["Kn","Cm","Ap","An","Sn","Ev"]
PALETTE = ["tab:blue","tab:orange","tab:green","tab:red","tab:purple","tab:brown"]
CMAP    = ListedColormap(PALETTE)

_TEXT_COL_CANDIDATES = [
    "verb","Verb","VERB",
    "token","Token",
    "word","Word",
    "lemma","Lemma",
    "term","Term",
]

def _find_text_series(dfx: pd.DataFrame) -> pd.Series:
    # 1) common column names
    for c in _TEXT_COL_CANDIDATES:
        if c in dfx.columns:
            print(f"[viz] using text column: {c}")
            return dfx[c].astype(str)
    # 2) patterns you already have in BT4
    if "AcceptedPattern" in dfx.columns:
        print(f"[viz] using text column: AcceptedPattern")
        return dfx["AcceptedPattern"].astype(str)
    if "SoftPattern" in dfx.columns:
        print(f"[viz] using text column: SoftPattern")
        return dfx["SoftPattern"].astype(str)
    # 3) ultimate fallback: synthetic labels (keeps plot working)
    print("[viz] no text-like column found; using synthetic labels")
    return pd.Series([f"item_{i}" for i in range(len(dfx))], index=dfx.index, dtype=str)

def _project_2d(vectors: np.ndarray):
    try:
        import umap
        reducer = umap.UMAP(n_components=2, random_state=42)
        Z = reducer.fit_transform(vectors)
        return Z, "UMAP"
    except Exception:
        from sklearn.decomposition import PCA
        Z = PCA(n_components=2, random_state=42).fit_transform(vectors)
        return Z, "PCA"

def visualize_model_fixed(
    df: pd.DataFrame,
    label_col: str = None,       # None -> prefer 'primary_effective' then 'PrimaryDomain'
    accepted_only: bool = True,  # filter to hard-accepted if column exists
    max_points_per_level: int = 800,
    title: str = "Bloom Model Visualization"
):
    # choose label column
    if label_col is None:
        label_col = "primary_effective" if "primary_effective" in df.columns else ("PrimaryDomain" if "PrimaryDomain" in df.columns else None)
    if label_col is None:
        raise KeyError("No label column found. Provide label_col or add 'primary_effective'/'PrimaryDomain' to the dataframe.")

    dfx = df.copy()

    # hard-accepted only (optional)
    if accepted_only and "Accepted" in dfx.columns:
        dfx = dfx[pd.to_numeric(dfx["Accepted"], errors="coerce").fillna(0).astype(int) == 1].copy()

    # keep only rows with labels in LEVELS
    labs = dfx[label_col].astype(str).str.strip()
    dfx = dfx[labs.isin(LEVELS)].copy()
    if dfx.empty:
        raise ValueError("No rows to plot after filtering (check Accepted filter or label_col values).")

    # balanced subsample for legibility
    parts = []
    for lv in LEVELS:
        g = dfx[dfx[label_col] == lv]
        if g.empty: 
            continue
        k = min(max_points_per_level, len(g))
        parts.append(g.sample(n=k, random_state=42))
    dfx = pd.concat(parts, ignore_index=True) if parts else dfx

    # ---- embeddings ----
    text_series = _find_text_series(dfx)
    words = text_series.tolist()

    # use your existing embed_words() if defined; otherwise TF-IDF+SVD fallback
    try:
        V = embed_words(words)  # should exist from earlier cells
    except Exception:
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.decomposition import TruncatedSVD
        from sklearn.preprocessing import Normalizer
        vec = TfidfVectorizer()
        X = vec.fit_transform(words)
        svd = TruncatedSVD(n_components=min(128, max(2, X.shape[0]-1)), random_state=42)
        V = Normalizer(copy=False).fit_transform(svd.fit_transform(X))

    Z, method = _project_2d(V)

    # class indices match LEVELS ordering
    y_idx = np.array([LEVELS.index(x) for x in dfx[label_col].astype(str)], dtype=int)

    # ---- plot ----
    plt.figure(figsize=(7.8,6.2))
    plt.scatter(Z[:,0], Z[:,1], c=y_idx, cmap=CMAP, vmin=0, vmax=len(LEVELS)-1, s=10, alpha=0.80)
    plt.title(f"{title} ({method}; backend={globals().get('_VECT_BACKEND','svd')})")
    plt.xticks([]); plt.yticks([])

    # consistent legend colors
    handles = [plt.Line2D([], [], marker='o', linestyle='', label=lv, markersize=6,
                          markerfacecolor=PALETTE[i], markeredgecolor='none')
               for i, lv in enumerate(LEVELS)]
    plt.legend(handles=handles, title="Primary domain", loc="lower left", frameon=True)

    out = FIG_DIR / "model_space_2d.png"
    plt.tight_layout(); plt.savefig(out, dpi=220); plt.close()
    print("Saved:", out)

# Run it (as-is PrimaryDomain, hard-accepted only)
visualize_model_fixed(df, label_col="PrimaryDomain", accepted_only=True, title="Bloom Model Visualization")


[viz] using text column: verb


C:\Users\talha4\AppData\Local\anaconda3\envs\bloom-nlp\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


Saved: D:\Downloads\Bloom-s-Taxonomy-extension-master\Bloom-s-Taxonomy-extension-master\results\figures\model_space_2d.png


In [13]:
# ===== Tables 7–9: centroid cosine distance + nominal gap means + Spearman trend (BT4-ready, no df_core needed) =====
import numpy as np, pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer
from scipy.stats import spearmanr

LEVELS = ["Kn","Cm","Ap","An","Sn","Ev"]

# ---------- helpers ----------
def pick_verb_col(df):
    for c in ["verb","Verb_clean","verb_clean","Verb","word"]:
        if c in df.columns: return c
    raise KeyError("No verb column found. Expected one of: verb, Verb_clean, verb_clean, Verb, word")

def pick_label_cols(df):
    # Prefer accept_* (model output)
    if all(f"accept_{lv}" in df.columns for lv in LEVELS):
        return {lv: f"accept_{lv}" for lv in LEVELS}, "accept_* (model output)"
    # Or direct Kn..Ev columns
    if all(lv in df.columns for lv in LEVELS):
        return {lv: lv for lv in LEVELS}, "Kn..Ev (in-file labels)"
    raise KeyError("Need either accept_Kn..accept_Ev or Kn..Ev columns in df.")

def expanded_lists_from_df(df, verb_col, label_cols):
    d = {lv: [] for lv in LEVELS}
    for _, r in df.iterrows():
        w = str(r[verb_col]).strip()
        if not w:
            continue
        for lv in LEVELS:
            if int(r[label_cols[lv]]) == 1:
                d[lv].append(w)
    return d

def centroid_cosine_distance_matrix(wordlists, mdl):
    """Return 6x6 cosine distance matrix between L2-normalized centroids (paper definition)."""
    d = mdl.get_sentence_embedding_dimension()
    C = np.zeros((6, d), dtype=float)

    for i, lv in enumerate(LEVELS):
        W = [str(w).strip() for w in wordlists.get(lv, []) if str(w).strip()]
        if len(W) == 0:
            C[i] = np.zeros(d)
            continue
        V = mdl.encode(W, normalize_embeddings=True, show_progress_bar=False)  # unit vectors
        c = V.mean(axis=0)
        c = c / (np.linalg.norm(c) + 1e-12)  # L2-normalize centroid
        C[i] = c

    D = np.zeros((6,6), float)
    for i in range(6):
        for j in range(i+1, 6):
            D[i,j] = D[j,i] = float(1.0 - np.dot(C[i], C[j]))  # cosine distance
    return D

def table7_df(D, title):
    print("\n" + title)
    df7 = pd.DataFrame(D, index=LEVELS, columns=LEVELS).round(6)
    display(df7)
    return df7

def nominal_gap_means(D):
    # mean of upper diagonal for each gap k=1..5
    rows = []
    for k in range(1, 6):
        vals = np.diag(D, k=k)
        rows.append({"k": k, "Dbar_k": float(np.mean(vals))})
    return pd.DataFrame(rows)

def spearman_one_tailed(df_nom):
    ks = df_nom["k"].to_numpy()
    ds = df_nom["Dbar_k"].to_numpy()
    rho, p_two = spearmanr(ks, ds)
    # one-tailed test for positive trend
    p_one = (p_two/2) if rho > 0 else (1 - p_two/2)
    return float(rho), float(p_one), float(p_two)

# ---------- REQUIRE: df (BT4_PROD) ----------
assert "df" in globals(), "Load your BT4_PROD table into a DataFrame named df first."

verb_col = pick_verb_col(df)
label_cols, label_mode = pick_label_cols(df)

print("Verb column:", verb_col)
print("Label mode:", label_mode)
print("Rows:", len(df), "| Unique verbs:", df[verb_col].nunique())

# ---------- Embedder: MPNet (768-D) ----------
mdl = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
print("Embedding model:", "sentence-transformers/all-mpnet-base-v2",
      "| dim:", mdl.get_sentence_embedding_dimension())

# ===== EXTENDED TABLES (always) =====
ext_lists = expanded_lists_from_df(df, verb_col, label_cols)
D_ext = centroid_cosine_distance_matrix(ext_lists, mdl)

_ = table7_df(D_ext, "Table 7 — Extended centroid cosine distance matrix D(ℓ,ℓ′)")

T8_ext = nominal_gap_means(D_ext)
print("\nTable 8 (part) — Extended mean centroid distance by Bloom gap k")
display(T8_ext.round(6))

rho_ext, p1_ext, p2_ext = spearman_one_tailed(T8_ext)
T9_ext = pd.DataFrame([{
    "Set": "Extended",
    "Spearman_rho": rho_ext,
    "p_one_tailed": p1_ext,
    "p_two_tailed": p2_ext
}])
print("\nTable 9 (part) — Extended monotonicity trend (Spearman)")
display(T9_ext.round(6))

# ===== CORE TABLES (optional if core lists exist in notebook) =====
core_lists = None

# Option A: wordlists exists (list of 6 lists)
if "wordlists" in globals():
    try:
        core_lists = {lv: list(lst) for lv, lst in zip(LEVELS, wordlists)}
    except Exception:
        core_lists = None

# Option B: individual lists exist
if core_lists is None:
    names = ["knowledge_words","comprehension_words","application_words",
             "analysis_words","synthesis_words","evaluation_words"]
    if all(n in globals() for n in names):
        core_lists = {
            "Kn": list(knowledge_words), "Cm": list(comprehension_words),
            "Ap": list(application_words), "An": list(analysis_words),
            "Sn": list(synthesis_words), "Ev": list(evaluation_words)
        }

if core_lists is None:
    print("\n[Core skipped] Core verb lists not found in this notebook (wordlists or knowledge_words..evaluation_words).")
    print("If you want Core vs Extended Tables 8–9, copy core lists into BT4 or run this cell in BT3 where core lists exist.")
else:
    D_core = centroid_cosine_distance_matrix(core_lists, mdl)
    _ = table7_df(D_core, "Core centroid cosine distance matrix (for Table 8/9)")

    T8_core = nominal_gap_means(D_core)
    print("\nTable 8 (part) — Core mean centroid distance by Bloom gap k")
    display(T8_core.round(6))

    rho_core, p1_core, p2_core = spearman_one_tailed(T8_core)
    T9_core = pd.DataFrame([{
        "Set": "Core",
        "Spearman_rho": rho_core,
        "p_one_tailed": p1_core,
        "p_two_tailed": p2_core
    }])
    print("\nTable 9 (part) — Core monotonicity trend (Spearman)")
    display(T9_core.round(6))

    # Final combined tables (paper-ready)
    Table8 = T8_core.merge(T8_ext, on="k", suffixes=("_core","_ext"))
    Table8.columns = ["k", "Dbar_k_core", "Dbar_k_ext"]
    print("\nTable 8 — Core vs Extended (paper-ready)")
    display(Table8.round(6))

    Table9 = pd.concat([T9_core, T9_ext], ignore_index=True)
    print("\nTable 9 — Trend stats (paper-ready)")
    display(Table9.round(6))


Verb column: verb
Label mode: accept_* (model output)
Rows: 1824 | Unique verbs: 1824
Embedding model: sentence-transformers/all-mpnet-base-v2 | dim: 768

Table 7 — Extended centroid cosine distance matrix D(ℓ,ℓ′)


,Kn,Cm,Ap,An,Sn,Ev
Kn,0.000000,0.138227,0.176605,0.150756,0.201175,0.233947
Cm,0.138227,0.000000,0.077985,0.073295,0.055995,0.132172
Ap,0.176605,0.077985,0.000000,0.052868,0.050172,0.099031
An,0.150756,0.073295,0.052868,0.000000,0.083313,0.115085
Sn,0.201175,0.055995,0.050172,0.083313,0.000000,0.124316
Ev,0.233947,0.132172,0.099031,0.115085,0.124316,0.000000



Table 8 (part) — Extended mean centroid distance by Bloom gap k


,k,Dbar_k
0,1,0.095342
1,2,0.103789
2,3,0.101927
3,4,0.166673
4,5,0.233947



Table 9 (part) — Extended monotonicity trend (Spearman)


...

,Set,Spearman_rho,p_one_tailed,p_two_tailed
0,Core,0.9,0.018693,0.037386



Table 8 — Core vs Extended (paper-ready)


,k,Dbar_k_core,Dbar_k_ext
0,1,0.063909,0.095342
1,2,0.054650,0.103789
2,3,0.081585,0.101927
3,4,0.097060,0.166673
4,5,0.124355,0.233947



Table 9 — Trend stats (paper-ready)


,Set,Spearman_rho,p_one_tailed,p_two_tailed
0,Core,0.9,0.018693,0.037386
1,Extended,0.9,0.018693,0.037386


In [14]:
# === Extension summary table (like Table 5) — shown in output ===
import pandas as pd
from IPython.display import display

rows = []

# 0..6 entries
for k in range(0, 7):
    rows.append({
        "Metric": f"# of verbs with {k} entries",
        "Value":  int(counts_by_entries.get(k, 0))
    })

# Entries acquired by domain
for lv in LEVELS:
    rows.append({
        "Metric": f"# of entries classified into {lv}",
        "Value":  int(per_domain_entries.get(lv, 0))
    })

# Global stats
rows.extend([
    {"Metric": "Acceptance rate",          "Value": float(acc_rate)},
    {"Metric": "Entry rate",               "Value": float(entry_rate)},
    {"Metric": "# of all input verbs (n)", "Value": int(n)},
])

tbl_ext_summary = pd.DataFrame(rows)
print("\nExtension summary (Table 5 style):")
display(tbl_ext_summary)



Extension summary (Table 5 style):


,Metric,Value
0,# of verbs with 0 entries,318.000000
1,# of verbs with 1 entries,1246.000000
2,# of verbs with 2 entries,185.000000
3,# of verbs with 3 entries,61.000000
4,# of verbs with 4 entries,10.000000
5,# of verbs with 5 entries,4.000000
6,# of verbs with 6 entries,0.000000
7,# of entries classified into Kn,49.000000
8,# of entries classified into Cm,373.000000
9,# of entries classified into Ap,574.000000
